# Multi-Head Self-Attention with NumPy
Pure NumPy implementation of head splitting, scaled dot-product attention, and multi-head attention.

In [1]:
import numpy as np

L, d_model, h = 3, 4, 2
X = np.array([[1., 0., 1., 0.],
              [0., 1., 0., 1.],
              [1., 1., 0., 0.]])
Wq = np.eye(4)
Wk = np.eye(4)
Wv = np.arange(16.).reshape(4, 4) / 10.
Wo = np.eye(4) * 2.

In [2]:
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

In [3]:
def split_heads(x, h):
    L, d_model = x.shape
    if d_model % h != 0:
        raise ValueError("d_model must be divisible by h")
    d_head = d_model // h
    return x.reshape(L, h, d_head).transpose(1, 0, 2)

In [4]:
def sdpa(Q, K, V):
    dk = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(dk)
    attn = softmax(scores, axis=-1)
    out = attn @ V
    return out, attn

In [5]:
def multi_head(X, Wq, Wk, Wv, Wo, h):
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    Qh = split_heads(Q, h)
    Kh = split_heads(K, h)
    Vh = split_heads(V, h)

    head_out, _ = sdpa(Qh, Kh, Vh)
    combined = head_out.transpose(1, 0, 2).reshape(X.shape[0], -1)
    return combined @ Wo

In [6]:
Q = np.zeros((2, 2))
K = np.zeros((4, 2))
V = np.array([[1., 1.], [2., 2.], [3., 3.], [4., 4.]])
out, attn = sdpa(Q, K, V)
print(np.round(attn[0], 3))
print(np.allclose(out, V.mean(0)))
print(split_heads(X, h).shape)
Y = multi_head(X, Wq, Wk, Wv, Wo, h)
print(Y.shape)

[0.25 0.25 0.25 0.25]
True
(2, 3, 2)
(3, 4)
